# Uniform segmentation stats

In [2]:
import os
import numpy as np
import pandas as pd

from anomaly.utils import specobjid_to_idx

from sdss.metadata import MetaData

meta = MetaData()

## Directories

In [3]:
phd_dir = "/home/elom/phd"
thesis_dir = f"{phd_dir}/thesis"
data_dir = f"{phd_dir}/code"
spectra_dir = f"{data_dir}/spectra"
scores_dir = f"{data_dir}/scores"
models_dir = f"{data_dir}/models"
latent_dir = f"{data_dir}/latent"
explanations_dir = f"{data_dir}/explanations"
ch_5_dir = f"{thesis_dir}/chapters/05_figures"

# Data

In [4]:
wave = np.load(f"{spectra_dir}/wave_spectra_imputed.npy")
wave_nm = wave * 0.1

spectra = np.load(f"{spectra_dir}/spectra_imputed.npy", mmap_mode="r")

final_meta_df = pd.read_csv(
    f"{spectra_dir}/final_spec_n_z_warning_drop.csv.gz",
    index_col="specobjid",
)

meta_with_ratios_df = pd.read_csv(
    f"{spectra_dir}/final_spec_n_z_warning_drop_with_ratios.csv.gz",
    index_col="specobjid",
)

idx_id_spec = np.load(f"{spectra_dir}/ids_imputing.npy", mmap_mode="r")
bin_id = "bin_03"
latent_arr = np.load(f"{latent_dir}/{bin_id}/latent_{bin_id}.npy", mmap_mode="r")

## N segments

In [ ]:
# n_segments = 110 # Segment width: 34, Last segment width: 33
# n_segments = 150 # Segment width: 25, Last segment width: 23
# n_segments = 163 # Segment width: 23, Last segment width: 24
# n_segments = 221 # Segment width: 17, Last segment width: 16
n_segments = 343  # Segment width: 11, Last segment width: 0
segment_width = wave.shape[0] // n_segments
last_segment_width = wave.shape[0] % n_segments
print(f"Segment width: {segment_width}, Last segment width: {last_segment_width}")

n_wave = wave.size
print(f"Total pixels: {n_wave}")

n_trim = n_wave * 0.03
n_trim_per_segment = n_trim / 17
print(f"Trimed pixels: {n_trim:.3f}, Trimmed in segments: {n_trim_per_segment:.3f}")

Segment width: 11, Last segment width: 0
Total pixels: 3773
Trimed pixels: 113.190, Trimmed in segments: 6.658


In [14]:
n_segments_list = range(1, 700, 1)

for n_segments in n_segments_list:

    segment_width = wave.shape[0] // n_segments
    last_segment_width = wave.shape[0] % n_segments
    segment_condition = abs(segment_width - last_segment_width) < 3
    segment_condition = segment_condition or last_segment_width == 0

    if segment_condition is True:

        segment_width_stat = (
            f"Segment width: {segment_width}, Last segment width: {last_segment_width}"
        )
        print(f"n_segments = {n_segments} # {segment_width_stat}")

        continue

n_segments = 1 # Segment width: 3773, Last segment width: 0
n_segments = 7 # Segment width: 539, Last segment width: 0
n_segments = 11 # Segment width: 343, Last segment width: 0
n_segments = 49 # Segment width: 77, Last segment width: 0
n_segments = 73 # Segment width: 51, Last segment width: 50
n_segments = 76 # Segment width: 49, Last segment width: 49
n_segments = 77 # Segment width: 49, Last segment width: 0
n_segments = 81 # Segment width: 46, Last segment width: 47
n_segments = 91 # Segment width: 41, Last segment width: 42
n_segments = 101 # Segment width: 37, Last segment width: 36
n_segments = 110 # Segment width: 34, Last segment width: 33
n_segments = 150 # Segment width: 25, Last segment width: 23
n_segments = 163 # Segment width: 23, Last segment width: 24
n_segments = 221 # Segment width: 17, Last segment width: 16
n_segments = 342 # Segment width: 11, Last segment width: 11
n_segments = 343 # Segment width: 11, Last segment width: 0
n_segments = 418 # Segment width: 9, 

In [14]:
w_min, w_max = wave_nm.min(), wave_nm.max()
print(f"Min wave: {w_min:.1f} nm\nMax wave: {w_max:.1f} nm")

Min wave: 369.8 nm
Max wave: 750.0 nm


In [15]:
total_pixels = len(wave_nm)
num_standard_segments = 343
seg_width = total_pixels // num_standard_segments
# 1. Create the pixel index boundaries for each segment
segment_bounds = []
for i in range(num_standard_segments):
    start_idx = i * seg_width
    end_idx = start_idx + seg_width  # Non-inclusive upper bound
    segment_bounds.append((start_idx, end_idx))


segment_bounds.append((num_standard_segments * seg_width, total_pixels))

# 2. Extract physical metrics for each segment
segment_data = []
for seg_id, (start, end) in enumerate(segment_bounds):
    if start == n_wave:
        continue
    # Wavelength at the start and just before the end index
    lambda_start = wave_nm[start]
    lambda_end = wave_nm[end - 1]

    # Physical delta (width) of this specific segment
    delta_lambda = lambda_end - lambda_start

    segment_data.append(
        {
            "segment_id": seg_id,
            "start_pixel": start,
            "end_pixel": end - 1,
            "lambda_start": lambda_start,
            "lambda_end": lambda_end,
            "width_nm": delta_lambda,
        }
    )

# 3. Analyze the results
widths = [seg["width_nm"] for seg in segment_data]
# Exclude the last segment for a fair standard width calculation
mean_width = np.mean(widths[:-1])
median_width = np.median(widths[:-1])
min_width = np.min(widths[:-1])
max_width = np.max(widths[:-1])
std_width = np.std(widths[:-1])

print(f"Total Segments: {len(segment_data)}")
print(f"Mean pixel segment width: {mean_width:.2f} nm")
print(f"Median pixel segment width: {median_width:.2f} nm")
print(f"Min pixel segment width:  {min_width:.2f} nm")
print(f"Max pixel segment width:  {max_width:.2f} nm")
print(f"Std pixel segment width:  {std_width:.2f} nm")
print(f"Final segment width: {segment_data[-1]['width_nm']:.2f} nm")

width_threshold = 1.8
print(f"Segments with width greater than {width_threshold} nm:")

for seg in segment_data:
    if seg["width_nm"] > width_threshold:
        print(
            f"Segment {seg['segment_id']}: "
            f"Range: {seg['lambda_start']:.1f} to {seg['lambda_end']:.1f} nm"
        )

Total Segments: 343
Mean pixel segment width: 1.01 nm
Median pixel segment width: 1.00 nm
Min pixel segment width:  1.00 nm
Max pixel segment width:  1.70 nm
Std pixel segment width:  0.06 nm
Final segment width: 1.00 nm
Segments with width greater than 1.8 nm:
